In [1]:
from gaitExtraction import gait_extraction
import gaitFeaturesExtraction
import handFeaturesExtraction
import os
import re
import json
import numpy as np
import glob
import joblib
import cv2

<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
/home/leo/miniconda3/envs/pdmodel/lib/python3.8/site-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '


In [2]:
# pth_2d = "/HDD3/Leo/Pose2D/ViTPose/TW_gait_2D_pose_data/processed_20200528_2ABC.npy"
# pth_3d = "/HDD3/Leo/Pose2D/TW_gait_3D_pose_data/processed_20200528_2ABC.npz"

# gait_feature = gaitFeaturesExtraction.pose_features_extract(pth_2d, pth_3d,  plot_results=True, save_fig_pth="./Test/gait_test.png")


In [3]:
path3 = r'../../handOutput3/*_A*_hand.txt'
files3_ls = glob.glob(path3)

pid_ls = []

json_file = 'handdata_20250605.json'
result_dt = {}
error_ls = []

for filepth in files3_ls:
    date = filepth.split("_")[0].split("/")[-1]
    pid = filepth.split("_")[1]

    l_hand = f"../../handOutput3/{date}_{pid}_AL_hand.txt"
    r_hand = f"../../handOutput3/{date}_{pid}_AR_hand.txt"

    if os.path.isfile(l_hand) & os.path.isfile(r_hand):
        pid_ls.append((date,pid))

pid_ls = set(pid_ls)

for i , all_id in enumerate(pid_ls):
    date = all_id[0]
    pid = all_id[1]

    pid_name = f"{date}_{pid}"

    l_hand = f"../../handOutput3/{date}_{pid}_AL_hand.txt"
    r_hand = f"../../handOutput3/{date}_{pid}_AR_hand.txt"

    video_path = f"/HDD3/leo/Data/PD/PD_Data_Hand/{date}/{date}_{pid}AR.mp4"
    
    if not os.path.isfile(video_path):
        fps = 59
    else:
        cap = cv2.VideoCapture(video_path)
        fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    try:
        arr = handFeaturesExtraction.single_thumb_index_hand(r_hand, l_hand, f"./Test/Hand/{date}_{pid}_", fps=fps)
        result_dt[pid_name] = arr
        
        # Save after each successful extraction
        with open(json_file, 'w') as f:
            json.dump(result_dt, f, indent=2)
            
    except Exception as e:
                print(f"Error processing {pid_name}: {e}")
                error_ls.append(pid_name)
    


In [3]:
ls_2d = []
ls_3d = []

for r, f, files in os.walk("/HDD3/Leo/Pose2D/ViTPose/TW_gait_2D_pose_data/"):
    for file in files:
        if file[-4:] == ".npy":
            ls_2d.append(file)
            
for r, f, files in os.walk("/HDD3/Leo/Pose2D/TW_gait_3D_pose_data/"):
    for file in files:
        if file[-4:] == ".npz":
            ls_3d.append(file)
            
pth_2d = "/HDD3/Leo/Pose2D/ViTPose/TW_gait_2D_pose_data/"
pth_3d = "/HDD3/Leo/Pose2D/TW_gait_3D_pose_data/"
json_file = 'gaitdata_20250605.json'
result_dt = {}
error_ls = []

for file2d in ls_2d:
    for file3d in ls_3d:
        if file2d[:-4] in file3d:
            file_3d = file3d
    file_pth_2d = f"{pth_2d}{file2d}"
    file_pth_3d = f"{pth_3d}{file_3d}"
    full_id = "_".join(file2d[:-4].split("_")[1:])
    # Extract only date_digits, e.g., 20200611_20
    
    match = re.match(r'^(\d+_\d+)', full_id)
    pid_name = match.group(1) if match else full_id
    
    try:
        gait_feature = gaitFeaturesExtraction.pose_features_extract(file_pth_2d, file_pth_3d,  plot_results=True, save_fig_pth=f"./Test/{pid_name}.png")
        result_dt[pid_name] = gait_feature
        
        # Save after each successful extraction
        with open(json_file, 'w') as f:
            json.dump(result_dt, f, indent=2)
            
    except Exception as e:
                print(f"Error processing {pid_name}: {e}")
                error_ls.append(pid_name)

In [6]:
json_file = 'gaitdata_20250605.json'
with open(json_file, 'r') as f:
    dt = json.load(f)
len(dt)

454

454